# OVM feasibility study

>Does OVM-guided beam search improve GSM8K accuracy and final-candidate quality over unguided sampling at K=20, and at what runtime cost?

* Checkpoints from Mistral-7B OVM on Hugging Face
* Pilot study with 100 questions on GSM8K
* Single A100 80GBs GPU


### Mount Drive

In [1]:
from google.colab import drive

drive.mount("drive", force_remount=True)

Mounted at drive


#### Point to the OVM folder that contains all configs, data and files

In [2]:
%cd drive/MyDrive/Heidelberg/process_reward_modeling/ovm/
!ls

/content/drive/MyDrive/Heidelberg/process_reward_modeling/ovm
analyze_results.py	 helpers.py		__pycache__
configs			 output			requirements-colab.txt
data			 ovm_feasibility.ipynb	run_pilot.py
download_models.py	 ovm-models
FreedomIntelligence_OVM  prepare_pilot.py


### Install the requirements and additional/heavier dependencies

- requires session restart

In [ ]:
!pip install -r requirements-colab.txt
## may not be needed
# !pip install torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 126.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 12.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 kB 33.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 435.4/435.4 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 152.6 MB/s eta 0:00:00
  C

### Download and load models

 - downloads the generator and the verifier
 - writes the manifest
 - allow some time for the run to complete
 - capture the outputs -- many frequent updates may crash the Colab session, check whether the manifest exists to ensure download is successful
 - not necessary to do in the same session, as long as the weights are downloaded

In [ ]:
%%capture
!python3 download_models.py --output ovm-models

### Prepare Pilot Study dataset

- generates the pilot questions (GSM8K 100 questions)
- writes the manifest

In [ ]:
!python3 prepare_pilot.py --config configs/ovm_feasibility.json --output-dir data/feasibility

Pilot: data/feasibility/test.jsonl, Manifest: data/feasibility/pilot_manifest.json


### Run Pilot Study

- first by running a smoke test -- small run on a limited number of questions to ensure the setup is fully correct
- full run
- each run starts with a `warmup_count` example passes to initialize models, kernels, etc. and ensure fair comparison


#### Smoke Test

In [ ]:
!python run_pilot.py --config configs/ovm_feasibility.json --model-snapshot ovm-models --pilot-manifest data/feasibility/pilot_manifest.json --output-dir output/ovm-smoke --seeds 42 --limit 2

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
GPU: NVIDIA A100-SXM4-80GB with 79.250732421875 GiB memory on device: cuda:0
Loading Generator and Verifier checkpoints...
+ [Model] Initializing L

##### Full Run
- ~3 hours for 3 seeds

In [ ]:
!python run_pilot.py --config configs/ovm_feasibility.json --model-snapshot ovm-models --pilot-manifest data/feasibility/pilot_manifest.json --output-dir output/ovm --seeds 41 42 43

/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
/usr/local/lib/python3.13/dist-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/usr/local/lib/python3.13/dist-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
GPU: NVIDIA A100-SXM4-80GB with 79.250732421875 GiB memory on device: cuda:0
Loading Generator and Verifier checkpoints...
+ [Model] Initializing L

#### Analysis

* the analysis ideally gives clear answers to the requirements (custom-defined) of the feasibility study
* **requirements**

```
  "feasibility_gates": {
    "minimum_completed_seeds": 3,
    "minimum_ovm_minus_self_consistency_accuracy": 0.05,
    "require_no_seed_worse_than_self_consistency": true,
    "minimum_ovm_minus_greedy_accuracy": 0.15,
    "minimum_last_expansion_minus_vanilla_correct_fraction": 0.15,
    "minimum_ovm_minus_orm_accuracy": -0.02,
    "maximum_ovm_to_sampling_runtime_ratio": 3.0
  }
```
* trying to somehow confirm the same trends observed in the original OVM paper

In [19]:
from IPython.display import Markdown, display
import os

!python3 analyze_results.py --config configs/ovm_feasibility.json --pilot-manifest data/feasibility/pilot_manifest.json --input-dir output/ovm --output-json output/ovm-eval.json --output-markdown output/ovm-eval.md


path_md = "output/ovm-eval.md"

if os.path.exists(path_md):

    with open(path_md, "r") as f:
        analysis_md = f.read()

    # display .md
    display(Markdown(analysis_md))

Feasibility Study Check: PASS


# OVM Feasibility Results

**Decision: PASS**

Pilot Study: 100 GSM8K questions x 3 seeds
## Average Accuracy


| Method | Accuracy |
|---:|---:|
| Greedy | 57.00% |
| Self-consistency (K=20) | 62.67% |
| ORM post-selection (K=20) | 79.67% |
| OVM guided (K=20, b=10) | 80.33% |
## Seed accuracy


| Seed | Greedy | SC | ORM | OVM |
|---:|---:|---:|---:|---:|
| 41 | 57.00% | 61.00% | 83.00% | 79.00% |
| 42 | 57.00% | 64.00% | 78.00% | 80.00% |
| 43 | 57.00% | 63.00% | 78.00% | 82.00% |
## Feasibility gates


| Requirement | Observed | Threshold | Result |
|---|---:|---:|:---:|
| `minimum_completed_seeds` | 3 | 3 | PASS |
| `ovm_minus_self_consistency_accuracy` | 0.1767 | 0.05 | PASS |
| `no_seed_worse_than_self_consistency` | 0.16 | 0 | PASS |
| `ovm_minus_greedy_accuracy` | 0.2333 | 0.15 | PASS |
| `last_expansion_minus_vanilla_correct_fraction` | 0.2772 | 0.15 | PASS |
| `ovm_minus_orm_accuracy` | 0.006667 | -0.02 | PASS |
| `ovm_to_sampling_runtime_ratio` | 0.6895 | 3 | PASS |
| `maximum_peak_allocated_gib` | 38.9 | 78 | PASS |

## Observability

- Vanilla sample correct fraction: 48.25%
- OVM last-expansion correct fraction: 75.97%
- OVM final-beam correct fraction: 79.07%
- OVM / sampling runtime ratio: 0.69
- Maximum peak allocated CUDA memory: 38.90 GiB
